# SpaceX Falcon 9 — Data Collection with the SpaceX REST API

**Objective:** request launch data from the SpaceX REST API, parse the JSON response, enrich each launch with booster, launch site, payload and core information, keep only Falcon 9 launches, and handle missing values.

> Runs in **JupyterLite**. If the live API is unreachable from the browser, the notebook falls back to the dataset published by IBM so the rest of the analysis still runs.

In [ ]:
import piplite
await piplite.install(['numpy', 'pandas'])

In [ ]:
import pandas as pd
import numpy as np
import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
from js import fetch
import io, json

async def get_text(url):
    resp = await fetch(url)
    if not resp.ok:
        raise Exception(f"HTTP {resp.status} for {url}")
    return await resp.text()

async def get_json(url):
    return json.loads(await get_text(url))

async def get_csv(url):
    return pd.read_csv(io.StringIO(await get_text(url)))

## Helper functions
Each function takes the launch dataframe and calls a specific API endpoint using the IDs stored in each launch.

In [ ]:
BASE = "https://api.spacexdata.com/v4/"

async def getBoosterVersion(data):
    for x in data['rocket']:
        if x:
            response = await get_json(BASE + "rockets/" + str(x))
            BoosterVersion.append(response['name'])

async def getLaunchSite(data):
    for x in data['launchpad']:
        if x:
            response = await get_json(BASE + "launchpads/" + str(x))
            Longitude.append(response['longitude'])
            Latitude.append(response['latitude'])
            LaunchSite.append(response['name'])

async def getPayloadData(data):
    for load in data['payloads']:
        if load:
            response = await get_json(BASE + "payloads/" + load)
            PayloadMass.append(response['mass_kg'])
            Orbit.append(response['orbit'])

async def getCoreData(data):
    for core in data['cores']:
        if core['core'] is not None:
            response = await get_json(BASE + "cores/" + core['core'])
            Block.append(response['block'])
            ReusedCount.append(response['reuse_count'])
            Serial.append(response['serial'])
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])

## Task 1: Request and parse the SpaceX launch data
We use the static snapshot of the `/v4/launches/past` response provided by IBM so results are consistent with the course.

In [ ]:
static_json_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json"
launches_json = await get_json(static_json_url)
print("Launches received:", len(launches_json))

data = pd.json_normalize(launches_json)
data.head()

### Keep only the needed columns and clean multi-core / multi-payload launches

In [ ]:
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

# Remove rows with multiple cores (Falcon Heavy) and multiple payloads
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]

# Each list has a single element -> extract it
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])

# Convert date_utc to date and restrict the period
data['date'] = pd.to_datetime(data['date_utc']).dt.date
data = data[data['date'] <= datetime.date(2020, 11, 13)]
print(data.shape)

## Task 2: Enrich each launch with the API endpoints

In [ ]:
BoosterVersion, PayloadMass, Orbit, LaunchSite = [], [], [], []
Outcome, Flights, GridFins, Reused, Legs, LandingPad = [], [], [], [], [], []
Block, ReusedCount, Serial, Longitude, Latitude = [], [], [], [], []

api_ok = True
try:
    await getBoosterVersion(data)
    await getLaunchSite(data)
    await getPayloadData(data)
    await getCoreData(data)
except Exception as e:
    api_ok = False
    print("SpaceX API not reachable from this browser:", e)

if api_ok:
    launch_dict = {'FlightNumber': list(data['flight_number']),
                   'Date': list(data['date']),
                   'BoosterVersion': BoosterVersion,
                   'PayloadMass': PayloadMass,
                   'Orbit': Orbit,
                   'LaunchSite': LaunchSite,
                   'Outcome': Outcome,
                   'Flights': Flights,
                   'GridFins': GridFins,
                   'Reused': Reused,
                   'Legs': Legs,
                   'LandingPad': LandingPad,
                   'Block': Block,
                   'ReusedCount': ReusedCount,
                   'Serial': Serial,
                   'Longitude': Longitude,
                   'Latitude': Latitude}
    launch_data = pd.DataFrame(launch_dict)
    print("Built from the live API:", launch_data.shape)

## Task 3: Filter the dataframe to keep only Falcon 9 launches
(Fallback: if the API was blocked, load IBM's equivalent dataset of the API results before filtering.)

In [ ]:
if api_ok:
    data_falcon9 = launch_data[launch_data['BoosterVersion'] != 'Falcon 1'].copy()
    data_falcon9.loc[:, 'FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))
else:
    data_falcon9 = await get_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv")
    print("Loaded IBM dataset_part_1.csv (fallback)")

print(data_falcon9.shape)
data_falcon9.head()

## Task 4: Data wrangling — handle missing values

In [ ]:
data_falcon9.isnull().sum()

In [ ]:
# Replace missing PayloadMass with the mean (LandingPad keeps NaN = no landing pad used)
payload_mean = data_falcon9['PayloadMass'].mean()
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].replace(np.nan, payload_mean)
data_falcon9.isnull().sum()

In [ ]:
print("Falcon 9 launches:", data_falcon9.shape[0])
data_falcon9.to_csv('dataset_part_1.csv', index=False)

## Author

Jhoselyn Vera